<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/Random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import chi2_contingency
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving test_feature_engineered.parquet to test_feature_engineered.parquet
Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [ ]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [ ]:
#making 3 class return
# Train
conditions_train = [
    (train['Return'] > 3),
    (train['Return'] >= -3) & (train['Return'] <= 3),
    (train['Return'] < -3)
]

choices = [0, 1, 2]

train['Return_class_3'] = np.select(conditions_train, choices, default=np.nan)


# Validation
conditions_val = [
    (val['Return'] > 3),
    (val['Return'] >= -3) & (val['Return'] <= 3),
    (val['Return'] < -3)
]

val['Return_class_3'] = np.select(conditions_val, choices, default=np.nan)



In [ ]:
#making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_train = train["Return_class_3"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_val = val["Return_class_3"]


In [ ]:
#Dropping colums
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])

In [ ]:
# -----------------------------
# Combine train + validation
# -----------------------------
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all = X_all.copy()

cat_cols = X_all.select_dtypes(include=['object', 'category']).columns.tolist()

# -----------------------------
# Stratified K-Fold
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):

    print(f"\n--- Fold {fold} ---")

    X_tr = X_all.iloc[train_idx].copy()
    X_val_fold = X_all.iloc[val_idx].copy()

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    # -----------------------------
    # Encode categorical features
    # -----------------------------
    if cat_cols:
        encoder = OrdinalEncoder()

        X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
        X_val_fold[cat_cols] = encoder.transform(X_val_fold[cat_cols])

    # -----------------------------
    # Random Forest model
    # -----------------------------
    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    )

    # Train
    rf_model.fit(X_tr, y_tr)

    # Predict
    y_pred = rf_model.predict(X_val_fold)

    # Metrics
    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average="macro")

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print(classification_report(y_val_fold, y_pred))

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)

    reports.append(classification_report(y_val_fold, y_pred, output_dict=True))

# -----------------------------
# Average Results
# -----------------------------
print("\n===== AVERAGE RESULTS =====")
print("Average Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# -----------------------------
# Average classification report
# -----------------------------
avg_report = {}

for label in reports[0].keys():

    if label == "accuracy":
        avg_report[label] = np.mean([r[label] for r in reports])
        continue

    avg_report[label] = {}

    for metric in reports[0][label].keys():
        avg_report[label][metric] = np.mean([r[label][metric] for r in reports])

print("\nAverage Classification Report per class:")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
Accuracy: 0.5436807095343681
Macro F1: 0.48888467550371534
              precision    recall  f1-score   support

         0.0       0.32      0.64      0.43       411
         1.0       0.85      0.54      0.66      1474
         2.0       0.33      0.44      0.38       370

    accuracy                           0.54      2255
   macro avg       0.50      0.54      0.49      2255
weighted avg       0.67      0.54      0.57      2255


--- Fold 2 ---
Accuracy: 0.5532386867790594
Macro F1: 0.4980160843489063
              precision    recall  f1-score   support

         0.0       0.34      0.65      0.45       412
         1.0       0.85      0.55      0.67      1473
         2.0       0.33      0.45      0.38       369

    accuracy                           0.55      2254
   macro avg       0.51      0.55      0.50      2254
weighted avg       0.67      0.55      0.58      2254


--- Fold 3 ---
Accuracy: 0.5346051464063887
Macro F1: 0.47629045731817893
              

In [ ]:
# 29 core features
top_29_features = [
    "Surprise_Pct",
    "Sales/Stockholders Equity",
    "Payables Turnover",
    "Total Debt/EBITDA",
    "Sales/Stockholders Equity_QoQ",
    "Total Liabilities/Total Tangible Assets",
    "Long-term Debt/Book Equity",
    "Current Ratio",
    "Sales/Invested Capital",
    "Return on Equity",
    "Book/Market_Regime",
    "Price/Book_Regime",
    "Price/Operating Earnings (Basic, Excl. EI)_Regime",
    "Gross Profit/Total Assets",
    "P/E (Diluted, Incl. EI)_Regime",
    "Price/Operating Earnings (Diluted, Excl. EI)_Regime",
    "Pre-tax Return on Total Earning Assets",
    "Dividend Yield_Regime",
    "Effective Tax Rate",
    "Gross Profit Margin",
    "Interest/Average Long-term Debt",
    "Enterprise Value Multiple_Regime",
    "Inventory Turnover",
    "Receivables/Current Assets",
    "Cash Conversion Cycle (Days)",
    "Total Debt/Capital",
    "Interest Coverage Ratio",
    "Receivables Turnover",
    "Asset Turnover"
]

#Making new variable
X_train_29 = X_train[top_29_features].copy()
X_val_29 = X_val[top_29_features].copy()

In [ ]:
# ---------------------------------------------------------
# 29 features modelling
# ---------------------------------------------------------

X_all_29 = pd.concat([X_train_29, X_val_29], axis=0).copy()
y_all_29 = pd.concat([y_train, y_val], axis=0).copy()

cat_cols_29 = X_all_29.select_dtypes(include=['object', 'category']).columns.tolist()

# ---------------------------------------------------------
# 2. Stratified K-Fold
# ---------------------------------------------------------
skf_29 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list_29 = []
macro_f1_list_29 = []
reports_29 = []

for fold, (train_idx, val_idx) in enumerate(skf_29.split(X_all_29, y_all_29), 1):

    print(f"\n--- Model 29 Features | Fold {fold} ---")

    X_tr_29 = X_all_29.iloc[train_idx].copy()
    X_val_fold_29 = X_all_29.iloc[val_idx].copy()

    y_tr_29 = y_all_29.iloc[train_idx]
    y_val_fold_29 = y_all_29.iloc[val_idx]

    # Encoding
    if cat_cols_29:
        encoder_29 = OrdinalEncoder()
        X_tr_29[cat_cols_29] = encoder_29.fit_transform(X_tr_29[cat_cols_29])
        X_val_fold_29[cat_cols_29] = encoder_29.transform(X_val_fold_29[cat_cols_29])

    # ---------------------------------------------------------
    # 3. Random Forest model
    # ---------------------------------------------------------
    rf_model_29 = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    )

    # Train
    rf_model_29.fit(X_tr_29, y_tr_29)

    # Predict
    y_pred_29 = rf_model_29.predict(X_val_fold_29)


    acc_29 = accuracy_score(y_val_fold_29, y_pred_29)
    f1_29 = f1_score(y_val_fold_29, y_pred_29, average="macro")

    print(f"Accuracy: {acc_29:.4f} | Macro F1: {f1_29:.4f}")

    accuracy_list_29.append(acc_29)
    macro_f1_list_29.append(f1_29)
    reports_29.append(classification_report(y_val_fold_29, y_pred_29, output_dict=True))

# ---------------------------------------------------------
# 4. Average results for Model 29
# ---------------------------------------------------------
print("\n===== AVERAGE RESULTS MODEL 29 FEATURES =====")
avg_acc_29 = np.mean(accuracy_list_29)
avg_f1_29 = np.mean(macro_f1_list_29)
print(f"Average Accuracy 29: {avg_acc_29:.4f}")
print(f"Average Macro F1 29: {avg_f1_29:.4f}")

# Average classification report
avg_report_29 = {}
for label in reports_29[0].keys():
    if label == "accuracy":
        avg_report_29[label] = np.mean([r[label] for r in reports_29])
        continue
    avg_report_29[label] = {}
    for metric in reports_29[0][label].keys():
        avg_report_29[label][metric] = np.mean([r[label][metric] for r in reports_29])

df_report_29 = pd.DataFrame(avg_report_29).T
print("\nAverage Classification Report Model 29:")
print(df_report_29)


--- Model 29 Features | Fold 1 ---
Accuracy: 0.5472 | Macro F1: 0.4938

--- Model 29 Features | Fold 2 ---
Accuracy: 0.5563 | Macro F1: 0.5018

--- Model 29 Features | Fold 3 ---
Accuracy: 0.5319 | Macro F1: 0.4765

--- Model 29 Features | Fold 4 ---
Accuracy: 0.5590 | Macro F1: 0.5032

--- Model 29 Features | Fold 5 ---
Accuracy: 0.5674 | Macro F1: 0.5125

===== AVERAGE RESULTS MODEL 29 FEATURES =====
Average Accuracy 29: 0.5524
Average Macro F1 29: 0.4976

Average Classification Report Model 29:
              precision    recall  f1-score      support
0.0            0.336108  0.657749  0.444705   411.400000
1.0            0.864978  0.548603  0.671278  1473.200000
2.0            0.324159  0.450197  0.376704   369.600000
accuracy       0.552392  0.552392  0.552392     0.552392
macro avg      0.508415  0.552183  0.497562  2254.200000
weighted avg   0.679787  0.552392  0.581632  2254.200000


In [ ]:
# ---------------------------------------------------------
# 4. FEATURE IMPORTANCE
# ---------------------------------------------------------
importances = rf_model_29.feature_importances_
feature_names = X_all_29.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("\nTop 10 Belangrijkste Features volgens Random Forest:")
print(feature_importance_df.head(10))


Top 10 Belangrijkste Features volgens Random Forest:
                                              Feature  Importance
0                                        Surprise_Pct    0.192848
10                                 Book/Market_Regime    0.073849
16             Pre-tax Return on Total Earning Assets    0.073351
13                          Gross Profit/Total Assets    0.061533
9                                    Return on Equity    0.042855
5             Total Liabilities/Total Tangible Assets    0.038780
17                              Dividend Yield_Regime    0.037205
20                    Interest/Average Long-term Debt    0.033068
12  Price/Operating Earnings (Basic, Excl. EI)_Regime    0.032222
28                                     Asset Turnover    0.030528


In [ ]:
# Hyperparameter tuning for 29 features the best option
X_encoded_29 = X_all_29.copy()
y_all_29 = y_all_29.copy()

cat_cols_29 = X_encoded_29.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols_29:
    encoder_29 = OrdinalEncoder()
    X_encoded_29[cat_cols_29] = encoder_29.fit_transform(X_encoded_29[cat_cols_29])

# Sample weights for class imbalance

sample_weights_rf29 = compute_sample_weight(class_weight='balanced', y=y_all_29)

# Define Hyperparameter Search Space
param_dist_29 = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 6, 8, 10, None],
    'max_features': ['sqrt', 0.3, 0.5],
    'min_samples_leaf': [5, 10, 20],
    'min_samples_split': [10, 20],
    'max_samples': [0.7, 0.8],
    'bootstrap': [True]
}

rf_base_29 = RandomForestClassifier(random_state=42)

random_search_29 = RandomizedSearchCV(
    estimator=rf_base_29,
    param_distributions=param_dist_29,
    n_iter=25,
    cv=5,
    scoring='f1_macro',
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("Start Hyperparameter Tuning voor Random Forest 29...")
random_search_29.fit(X_encoded_29, y_all_29, sample_weight=sample_weights_rf29)

best_rf = random_search_29.best_params_

print("\n" + "="*50)
print("🎯 GEBRUIK DEZE INSTELLINGEN IN JE RF_MODEL_29 BLOK:")
print("="*50)
print(f"n_estimators      : {best_rf.get('n_estimators')}")
print(f"max_depth         : {best_rf.get('max_depth')}")
print(f"max_features      : {best_rf.get('max_features')}")
print(f"min_samples_leaf  : {best_rf.get('min_samples_leaf')}")
print(f"min_samples_split : {best_rf.get('min_samples_split')}")
print(f"max_samples       : {best_rf.get('max_samples')}")
print("-" * 50)
print(f"Beste Macro F1 tijdens Tuning: {random_search_29.best_score_:.4f}")
print("="*50)

Start Hyperparameter Tuning voor Random Forest 29...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

🎯 GEBRUIK DEZE INSTELLINGEN IN JE RF_MODEL_29 BLOK:
n_estimators      : 700
max_depth         : None
max_features      : 0.5
min_samples_leaf  : 20
min_samples_split : 20
max_samples       : 0.7
--------------------------------------------------
Beste Macro F1 tijdens Tuning: 0.4810


In [ ]:
# ---------------------------------------------------------
# 1. Data preparation
# ---------------------------------------------------------
X_all_29 = pd.concat([X_train_29, X_val_29], axis=0).copy()
y_all_29 = pd.concat([y_train, y_val], axis=0).copy()

cat_cols_29 = X_all_29.select_dtypes(include=['object', 'category']).columns.tolist()

# ---------------------------------------------------------
# 2. Encoding
# ---------------------------------------------------------
if cat_cols_29:
    encoder_29 = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_all_29[cat_cols_29] = encoder_29.fit_transform(X_all_29[cat_cols_29])

# ---------------------------------------------------------
# 3. Stratified K-Fold setup
# ---------------------------------------------------------
skf_29 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list_29 = []
macro_f1_list_29 = []
reports_29 = []

# ---------------------------------------------------------
# 4. Cross-validation loop
# ---------------------------------------------------------
for fold, (train_idx, val_idx) in enumerate(skf_29.split(X_all_29, y_all_29), 1):

    print(f"\n--- RF Model 29 Features | Fold {fold} ---")

    X_tr_29 = X_all_29.iloc[train_idx]
    X_val_fold_29 = X_all_29.iloc[val_idx]

    y_tr_29 = y_all_29.iloc[train_idx]
    y_val_fold_29 = y_all_29.iloc[val_idx]

    # ---------------------------------------------------------
    # Sample weights (ONLY imbalance correction used)
    # ---------------------------------------------------------
    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_tr_29
    )

    # ---------------------------------------------------------
    # Random Forest model (tuned settings)
    # ---------------------------------------------------------
    rf_model_29 = RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        max_features=0.5,
        min_samples_leaf=20,
        min_samples_split=20,
        max_samples=0.7,
        random_state=42,
        n_jobs=-1
    )

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------
    rf_model_29.fit(
        X_tr_29,
        y_tr_29,
        sample_weight=sample_weights
    )

    # ---------------------------------------------------------
    # Prediction
    # ---------------------------------------------------------
    y_pred_29 = rf_model_29.predict(X_val_fold_29)

    acc_29 = accuracy_score(y_val_fold_29, y_pred_29)
    f1_29 = f1_score(y_val_fold_29, y_pred_29, average="macro")

    print(f"Accuracy: {acc_29:.4f} | Macro F1: {f1_29:.4f}")

    accuracy_list_29.append(acc_29)
    macro_f1_list_29.append(f1_29)
    reports_29.append(classification_report(y_val_fold_29, y_pred_29, output_dict=True))

# ---------------------------------------------------------
# 5. Final CV results
# ---------------------------------------------------------
print("\n===== AVERAGE RESULTS RANDOM FOREST MODEL 29 =====")
print(f"Average Accuracy: {np.mean(accuracy_list_29):.4f}")
print(f"Average Macro F1: {np.mean(macro_f1_list_29):.4f}")

# ---------------------------------------------------------
# 6. Average classification report
# ---------------------------------------------------------
avg_report_29 = {}

for label in reports_29[0].keys():
    if label == "accuracy":
        avg_report_29[label] = np.mean([r[label] for r in reports_29])
        continue

    avg_report_29[label] = {}
    for metric in reports_29[0][label].keys():
        avg_report_29[label][metric] = np.mean([r[label][metric] for r in reports_29])

df_report_29 = pd.DataFrame(avg_report_29).T

print("\nAverage Classification Report Model 29:")
print(df_report_29)


--- RF Model 29 Features | Fold 1 ---
Accuracy: 0.6071 | Macro F1: 0.5328

--- RF Model 29 Features | Fold 2 ---
Accuracy: 0.6083 | Macro F1: 0.5303

--- RF Model 29 Features | Fold 3 ---
Accuracy: 0.5972 | Macro F1: 0.5167

--- RF Model 29 Features | Fold 4 ---
Accuracy: 0.6242 | Macro F1: 0.5394

--- RF Model 29 Features | Fold 5 ---
Accuracy: 0.6167 | Macro F1: 0.5377

===== AVERAGE RESULTS RANDOM FOREST MODEL 29 =====
Average Accuracy: 0.6107
Average Macro F1: 0.5314

Average Classification Report Model 29:
              precision    recall  f1-score      support
0.0            0.373153  0.569273  0.450519   411.400000
1.0            0.844597  0.660062  0.740898  1473.200000
2.0            0.358331  0.459940  0.402789   369.600000
accuracy       0.610683  0.610683  0.610683     0.610683
macro avg      0.525360  0.563092  0.531402  2254.200000
weighted avg   0.678832  0.610683  0.632469  2254.200000


In [ ]:
# ---------------------------------------------------------
# 1. Data preparation
# ---------------------------------------------------------
X_all_29 = pd.concat([X_train_29, X_val_29], axis=0).copy()
y_all_29 = pd.concat([y_train, y_val], axis=0).copy()

cat_cols_29 = X_all_29.select_dtypes(include=['object', 'category']).columns.tolist()

# ---------------------------------------------------------
# 2. Encoding
# ---------------------------------------------------------
if cat_cols_29:
    encoder_29 = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_all_29[cat_cols_29] = encoder_29.fit_transform(X_all_29[cat_cols_29])

# ---------------------------------------------------------
# 3. Stratified K-Fold setup
# ---------------------------------------------------------
skf_29 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list_29 = []
macro_f1_list_29 = []
reports_29 = []

# NIEUW: Maak lege lijsten om de Out-of-Fold (OOF) resultaten in te verzamelen
all_true_labels_rf = []
all_pred_labels_rf = []

# ---------------------------------------------------------
# 4. Cross-validation loop
# ---------------------------------------------------------
for fold, (train_idx, val_idx) in enumerate(skf_29.split(X_all_29, y_all_29), 1):

    print(f"\n--- RF Model 29 Features | Fold {fold} ---")

    X_tr_29 = X_all_29.iloc[train_idx]
    X_val_fold_29 = X_all_29.iloc[val_idx]

    y_tr_29 = y_all_29.iloc[train_idx]
    y_val_fold_29 = y_all_29.iloc[val_idx]

    # Sample weights (ONLY imbalance correction used)
    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_tr_29
    )

    # Random Forest model (tuned settings)
    rf_model_29 = RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        max_features=0.5,
        min_samples_leaf=20,
        min_samples_split=20,
        max_samples=0.7,
        random_state=42,
        n_jobs=-1
    )

    # Training
    rf_model_29.fit(
        X_tr_29,
        y_tr_29,
        sample_weight=sample_weights
    )

    # Prediction
    y_pred_29 = rf_model_29.predict(X_val_fold_29)

    y_true_array = np.array(y_val_fold_29).astype(int)
    y_pred_array = np.array(y_pred_29).astype(int)

    acc_29 = accuracy_score(y_true_array, y_pred_array)
    f1_29 = f1_score(y_true_array, y_pred_array, average="macro", labels=[0, 1, 2])

    print(f"Accuracy: {acc_29:.4f} | Macro F1: {f1_29:.4f}")

    accuracy_list_29.append(acc_29)
    macro_f1_list_29.append(f1_29)
    reports_29.append(classification_report(y_true_array, y_pred_array, output_dict=True, labels=[0, 1, 2]))

    all_true_labels_rf.extend(y_true_array)
    all_pred_labels_rf.extend(y_pred_array)

# ---------------------------------------------------------
# 5. Final CV results
# ---------------------------------------------------------
print("\n===== AVERAGE RESULTS RANDOM FOREST MODEL 29 =====")
print(f"Average Accuracy: {np.mean(accuracy_list_29):.4f}")
print(f"Average Macro F1: {np.mean(macro_f1_list_29):.4f}")

# ---------------------------------------------------------
# 6. Average classification report
# ---------------------------------------------------------
avg_report_29 = {}

for label in reports_29[0].keys():
    if label == "accuracy":
        avg_report_29[label] = np.mean([r[label] for r in reports_29])
        continue

    avg_report_29[label] = {}
    for metric in reports_29[0][label].keys():
        avg_report_29[label][metric] = np.mean([r[label][metric] for r in reports_29])

df_report_29 = pd.DataFrame(avg_report_29).T

print("\nAverage Classification Report Model 29:")
print(df_report_29)



class_names = ['Sell (2)', 'Neutral (1)', 'Buy (0)']

cm_rf = confusion_matrix(all_true_labels_rf, all_pred_labels_rf, labels=[0, 1, 2])

plt.figure(figsize=(7, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.title('Confusion Matrix - Random Forest (29 Features with Sample Weights)', fontsize=12, pad=15)
plt.ylabel('Actual Class (True Labels)', fontsize=11, labelpad=10)
plt.xlabel('Predicted Class (Model Predictions)', fontsize=11, labelpad=10)

plt.tight_layout()

plt.savefig('confusion_matrix_random_forest.png', dpi=300)
plt.show()

NameError: name 'pd' is not defined

In [ ]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving test_feature_engineered.parquet to test_feature_engineered.parquet


In [ ]:
test = pd.read_parquet("test_feature_engineered.parquet")

In [ ]:
#making 3 class return
# test
conditions_test = [
    (test['Return'] > 3),
    (test['Return'] >= -3) & (test['Return'] <= 3),
    (test['Return'] < -3)
]

choices = [0, 1, 2]

test['Return_class_3'] = np.select(conditions_test, choices, default=np.nan)

#making a return class
target = "Return"

X_test = test.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_test = test["Return_class_3"]

#Dropping colums
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_test = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns])

# 29 core features
top_29_features = [
    "Surprise_Pct",
    "Sales/Stockholders Equity",
    "Payables Turnover",
    "Total Debt/EBITDA",
    "Sales/Stockholders Equity_QoQ",
    "Total Liabilities/Total Tangible Assets",
    "Long-term Debt/Book Equity",
    "Current Ratio",
    "Sales/Invested Capital",
    "Return on Equity",
    "Book/Market_Regime",
    "Price/Book_Regime",
    "Price/Operating Earnings (Basic, Excl. EI)_Regime",
    "Gross Profit/Total Assets",
    "P/E (Diluted, Incl. EI)_Regime",
    "Price/Operating Earnings (Diluted, Excl. EI)_Regime",
    "Pre-tax Return on Total Earning Assets",
    "Dividend Yield_Regime",
    "Effective Tax Rate",
    "Gross Profit Margin",
    "Interest/Average Long-term Debt",
    "Enterprise Value Multiple_Regime",
    "Inventory Turnover",
    "Receivables/Current Assets",
    "Cash Conversion Cycle (Days)",
    "Total Debt/Capital",
    "Interest Coverage Ratio",
    "Receivables Turnover",
    "Asset Turnover"
]

#Making new variable
X_test_29 = X_test[top_29_features].copy()

In [ ]:
# 1. Preparing data
# ---------------------------------------------------------
X_all_29 = pd.concat([X_train_29, X_val_29], axis=0).copy()
y_all_29 = pd.concat([y_train, y_val], axis=0).copy()
X_test_final = X_test_29.copy()

cat_cols_29 = X_all_29.select_dtypes(include=['object', 'category']).columns.tolist()

# ---------------------------------------------------------
# 2. Encoding
# ---------------------------------------------------------
if cat_cols_29:
    encoder_final = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_all_29[cat_cols_29] = encoder_final.fit_transform(X_all_29[cat_cols_29])
    X_test_final[cat_cols_29] = encoder_final.transform(X_test_final[cat_cols_29])

# ---------------------------------------------------------
# 3. Training
# ---------------------------------------------------------
print("Training final Random Forest model on all development data...")

rf_final_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    class_weight="balanced"
)

rf_final_model.fit(X_all_29, y_all_29)

# ---------------------------------------------------------
# 4. Predicting
# ---------------------------------------------------------
y_test_pred_rf = rf_final_model.predict(X_test_final)
y_test_proba_rf = rf_final_model.predict_proba(X_test_final)

# ---------------------------------------------------------
# 5. Results
# ---------------------------------------------------------
print("\n" + "="*50)
print("=== FINAL RANDOM FOREST RESULTS ON TEST SET ===")
print("="*50)

test_acc_rf = accuracy_score(y_test, y_test_pred_rf)
test_f1_rf = f1_score(y_test, y_test_pred_rf, average='macro')

print(f"Test Accuracy: {test_acc_rf:.4f}")
print(f"Test Macro F1: {test_f1_rf:.4f}")
print("\nTest Classification Report (Random Forest):")
print(classification_report(y_test, y_test_pred_rf))

Training final Random Forest model on all development data...

=== FINAL RANDOM FOREST RESULTS ON TEST SET ===
Test Accuracy: 0.5464
Test Macro F1: 0.5355

Test Classification Report (Random Forest):
              precision    recall  f1-score   support

         0.0       0.48      0.53      0.51       400
         1.0       0.66      0.61      0.63       540
         2.0       0.47      0.47      0.47       352

    accuracy                           0.55      1292
   macro avg       0.54      0.54      0.54      1292
weighted avg       0.55      0.55      0.55      1292



In [ ]:
# 1.Preparing
# ---------------------------------------------------------
X_all_29 = pd.concat([X_train_29, X_val_29], axis=0).copy()
y_all_29 = pd.concat([y_train, y_val], axis=0).copy()
X_test_final = X_test_29.copy()

cat_cols_29 = X_all_29.select_dtypes(include=['object', 'category']).columns.tolist()

# ---------------------------------------------------------
# 2. Encoding
# ---------------------------------------------------------
if cat_cols_29:
    encoder_final = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_all_29[cat_cols_29] = encoder_final.fit_transform(X_all_29[cat_cols_29])
    X_test_final[cat_cols_29] = encoder_final.transform(X_test_final[cat_cols_29])

# ---------------------------------------------------------
# 3. Sample Weights
# ---------------------------------------------------------
sample_weights_rf29 = compute_sample_weight(class_weight='balanced', y=y_all_29)

# ---------------------------------------------------------
# 4. Train The model
# ---------------------------------------------------------
print("Training final Tuned Random Forest model on all development data...")

rf_final_model_tuned = RandomForestClassifier(
    n_estimators=700,           # Verhoogd van 300
    max_depth=None,             # Volledige groei (gecontroleerd door samples_leaf)
    max_features=0.5,           # Gebruik 50% van features per split
    min_samples_leaf=20,        # Voorkomt overfitting op individuele uitschieters
    min_samples_split=20,       # Minimum aantal samples om te splitsen
    max_samples=0.7,            # Gebruik 70% van data per boom (Bootstrapping)
    random_state=42,
    n_jobs=-1
)


rf_final_model_tuned.fit(X_all_29, y_all_29, sample_weight=sample_weights_rf29)

# ---------------------------------------------------------
# 5. Predicting
# ---------------------------------------------------------
y_test_pred_rf = rf_final_model_tuned.predict(X_test_final)
y_test_proba_rf = rf_final_model_tuned.predict_proba(X_test_final)

# ---------------------------------------------------------
# 6. Results
# ---------------------------------------------------------
print("\n" + "="*50)
print("=== FINAL TUNED RANDOM FOREST RESULTS ON TEST SET ===")
print("="*50)

test_acc_rf = accuracy_score(y_test, y_test_pred_rf)
test_f1_rf = f1_score(y_test, y_test_pred_rf, average='macro')

print(f"Test Accuracy: {test_acc_rf:.4f}")
print(f"Test Macro F1: {test_f1_rf:.4f}")
print("\nTest Classification Report (Tuned RF):")
print(classification_report(y_test, y_test_pred_rf))

Training final Tuned Random Forest model on all development data...

=== FINAL TUNED RANDOM FOREST RESULTS ON TEST SET ===
Test Accuracy: 0.5580
Test Macro F1: 0.5403

Test Classification Report (Tuned RF):
              precision    recall  f1-score   support

         0.0       0.51      0.47      0.49       400
         1.0       0.64      0.68      0.66       540
         2.0       0.47      0.47      0.47       352

    accuracy                           0.56      1292
   macro avg       0.54      0.54      0.54      1292
weighted avg       0.55      0.56      0.56      1292



In [ ]:
# MCnemar test
overzicht_RF = pd.DataFrame({
    'Echte Waarde': y_test.astype(int),
    'Verwachte Waarde': y_test_pred_rf.astype(int),
    'Resultaat': ['GOED' if echte == verwacht else 'FOUT' for echte, verwacht in zip(y_test, y_test_pred_rf)]
})

print(overzicht_RF.head(20))

overzicht_RF.to_csv('overzicht_rf_resultaten.csv', index=False)

     Echte Waarde  Verwachte Waarde Resultaat
36              0                 0      GOED
37              2                 0      FOUT
38              0                 0      GOED
39              2                 2      GOED
76              2                 0      FOUT
77              0                 2      FOUT
78              1                 0      FOUT
79              2                 2      GOED
116             1                 2      FOUT
117             2                 2      GOED
118             0                 2      FOUT
119             0                 0      GOED
156             1                 2      FOUT
157             1                 2      FOUT
158             2                 2      GOED
159             1                 2      FOUT
196             2                 2      GOED
197             0                 2      FOUT
198             2                 2      GOED
199             2                 2      GOED


In [ ]:
# ---------------------------------------------------------
# 1. How many hits
# ---------------------------------------------------------
analysis_df = test.copy()
analysis_df['Predicted_Class'] = y_test_pred_rf
analysis_df['Actual_Class'] = y_test.values

def get_stats(x):
    tips = x[x['Predicted_Class'] == 0]
    total = len(tips)
    if total == 0:
        return pd.Series({'Aantal Tips': 0, 'Hit-rate': 0, 'Zachte Miss': 0, 'Harde Miss': 0})

    hits = (tips['Actual_Class'] == 0).mean()
    soft = (tips['Actual_Class'] == 1).mean()
    hard = (tips['Actual_Class'] == 2).mean()

    return pd.Series({
        'Aantal Tips': total,
        'Hit-rate (Winst >3%)': f"{hits:.1%}",
        'Zachte Miss (-3% tot 3%)': f"{soft:.1%}",
        'Harde Miss (Verlies <-3%)': f"{hard:.1%}"
    })

# ---------------------------------------------------------
# 2. analysis
# ---------------------------------------------------------

print("\n--- RISICO ANALYSE: P/E REGIME ---")
pe_risico = analysis_df.groupby('P/E (Diluted, Incl. EI)_Regime').apply(get_stats).reset_index()
print(pe_risico)

print("\n--- RISICO ANALYSE: DIVIDEND YIELD REGIME ---")
div_risico = analysis_df.groupby('Dividend Yield_Regime').apply(get_stats).reset_index()
print(div_risico)

print("\n--- RISICO ANALYSE: Price-Book-Regime ---")
div_risico = analysis_df.groupby('Price/Book_Regime').apply(get_stats).reset_index()
print(div_risico)

print("\n--- RISICO ANALYSE: RETURN ON EQUITY (ROE) ---")
analysis_df['roe_bucket'] = pd.qcut(analysis_df['Return on Equity'].rank(method='first'), 5, labels=['Q1 (Laag)', 'Q2', 'Q3', 'Q4', 'Q5 (Hoog)'])
roe_risico = analysis_df.groupby('roe_bucket').apply(get_stats).reset_index()
print(roe_risico)

print("\n--- RISICO ANALYSE: CURRENT RATIO ---")
analysis_df['cr_bucket'] = pd.qcut(analysis_df['Current Ratio'].rank(method='first'), 5, labels=['Q1 (Laag)', 'Q2', 'Q3', 'Q4', 'Q5 (Hoog)'])
cr_risico = analysis_df.groupby('cr_bucket').apply(get_stats).reset_index()
print(cr_risico)

print("\n--- RISICO ANALYSE: SCHULDGRAAD (DEBT/EBITDA) ---")
analysis_df['debt_bucket'] = pd.qcut(analysis_df['Total Debt/EBITDA'].rank(method='first'), 5, labels=['Q1 (Laag)', 'Q2', 'Q3', 'Q4', 'Q5 (Hoog)'])
debt_risico = analysis_df.groupby('debt_bucket').apply(get_stats).reset_index()
print(debt_risico)


--- RISICO ANALYSE: P/E REGIME ---
  P/E (Diluted, Incl. EI)_Regime  Aantal Tips Hit-rate (Winst >3%)  \
0                            Low           14                64.3%   
1                     Medium_Low           60                50.0%   
2                    Medium_High           95                43.2%   
3                           High          106                50.0%   
4                        Extreme           98                58.2%   

  Zachte Miss (-3% tot 3%) Harde Miss (Verlies <-3%)  
0                    28.6%                      7.1%  
1                    20.0%                     30.0%  
2                    21.1%                     35.8%  
3                    20.8%                     29.2%  
4                    22.4%                     19.4%  

--- RISICO ANALYSE: DIVIDEND YIELD REGIME ---
  Dividend Yield_Regime  Aantal Tips Hit-rate (Winst >3%)  \
0                   Low          131                51.9%   
1            Medium_Low           71        

/tmp/ipykernel_4419/2623637615.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pe_risico = analysis_df.groupby('P/E (Diluted, Incl. EI)_Regime').apply(get_stats).reset_index()
/tmp/ipykernel_4419/2623637615.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pe_risico = analysis_df.groupby('P/E (Diluted, Incl. EI)_Regime').apply(get_stats).reset_index()
/tmp/ipykernel_4419/2623637615.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pan

In [ ]:
#ChiSquare test
def run_chi2_test(df, group_col):
    print(f"\n--- CHI-KWADRAAT TOETS VOOR: {group_col} ---")


    tips_only = df[df['Predicted_Class'] == 0]

    if tips_only.empty:
        print("Geen tips gevonden voor deze groep.")
        return

    contingency_table = pd.crosstab(tips_only[group_col], tips_only['Actual_Class'])

    chi2, p, dof, expected = chi2_contingency(contingency_table)

    print(f"Chi-square statistic: {chi2:.4f}")
    print(f"p-waarde: {p:.4f}")

    if p < 0.05:
        print("RESULTAAT: Statistisch Significant (p < 0.05). Het regime heeft invloed op de kwaliteit van de tips.")
    else:
        print("RESULTAAT: Niet significant (p >= 0.05). De verschillen zijn mogelijk toeval.")

    return p

test_cols = [
    'P/E (Diluted, Incl. EI)_Regime',
    'Dividend Yield_Regime',
    'Price/Book_Regime',
]

for col in test_cols:
    run_chi2_test(analysis_df, col)


--- CHI-KWADRAAT TOETS VOOR: P/E (Diluted, Incl. EI)_Regime ---
Chi-square statistic: 10.1694
p-waarde: 0.2533
RESULTAAT: Niet significant (p >= 0.05). De verschillen zijn mogelijk toeval.

--- CHI-KWADRAAT TOETS VOOR: Dividend Yield_Regime ---
Chi-square statistic: 10.9363
p-waarde: 0.2053
RESULTAAT: Niet significant (p >= 0.05). De verschillen zijn mogelijk toeval.

--- CHI-KWADRAAT TOETS VOOR: Price/Book_Regime ---
Chi-square statistic: 25.3625
p-waarde: 0.0013
RESULTAAT: Statistisch Significant (p < 0.05). Het regime heeft invloed op de kwaliteit van de tips.


In [ ]:
# 1. Value or Growth companies analysis
analysis_df = test.copy()
analysis_df['Predicted_Class'] = y_test_pred_rf
analysis_df['Actual_Class'] = y_test.values

# ---------------------------------------------------------
# 2. Define
# ---------------------------------------------------------
def assign_market_segment(row):
    pe = row['P/E (Diluted, Incl. EI)_Regime']
    bm = row['Price/Book_Regime']
    div = row['Dividend Yield_Regime']

    pe_low = ['Low', 'Medium Low'] # Voeg 'Extreme Low' toe als die bestaat
    pe_high = ['High', 'Extreme']

    bm_high = ['High', 'Extreme']
    bm_low = ['Low', 'Medium Low']

    div_high = ['High', 'Extreme']
    div_low = ['Low', 'Medium Low']

    value_score = 0
    if pe in pe_low: value_score += 1
    if bm in bm_high: value_score += 1
    if div in div_high: value_score += 1

    growth_score = 0
    if pe in pe_high: growth_score += 1
    if bm in bm_low: growth_score += 1
    if div in div_low: growth_score += 1

    if value_score >= 2:
        return 'Value (Maturity)'

    elif growth_score >= 2:
        return 'Growth (Expansion)'

    else:
        return 'Core / Mixed'

analysis_df['Market_Segment'] = analysis_df.apply(assign_market_segment, axis=1)

# ---------------------------------------------------------
# 3. Analyses
# ---------------------------------------------------------
def get_stats(x):
    tips = x[x['Predicted_Class'] == 0]
    total = len(tips)
    if total == 0:
        return pd.Series({'Aantal Tips': 0, 'Hit-rate': 0, 'Zachte Miss': 0, 'Harde Miss': 0})

    hits = (tips['Actual_Class'] == 0).mean()
    soft = (tips['Actual_Class'] == 1).mean()
    hard = (tips['Actual_Class'] == 2).mean()

    return pd.Series({
        'Aantal Tips': total,
        'Hit-rate (Winst >3%)': f"{hits:.1%}",
        'Zachte Miss (-3% tot 3%)': f"{soft:.1%}",
        'Harde Miss (Verlies <-3%)': f"{hard:.1%}"
    })

# ---------------------------------------------------------
# 4. Results
# ---------------------------------------------------------
print("\n--- ANALYSE PER MARKTSEGMENT (Interactie P/E, B/M, Dividend) ---")
segment_stats = analysis_df.groupby('Market_Segment').apply(get_stats).reset_index()
print(segment_stats)

# ---------------------------------------------------------
# 5. Significance
# ---------------------------------------------------------
print("\n--- CHI-KWADRAAT TOETS: SIGNIFICANTIE TUSSEN SEGMENTEN ---")

tips_only = analysis_df[analysis_df['Predicted_Class'] == 0]

if not tips_only.empty:
    contingency_table = pd.crosstab(tips_only['Market_Segment'], tips_only['Actual_Class'])

    chi2, p, dof, expected = chi2_contingency(contingency_table)

    print(contingency_table)
    print(f"\nChi-square statistic: {chi2:.4f}")
    print(f"p-waarde: {p:.4f}")

    if p < 0.05:
        print("RESULTAAT: Statistisch Significant. Het model presteert fundamenteel anders in de verschillende segmenten.")
    else:
        print("RESULTAAT: Niet significant. Er is geen statistisch bewijs dat het segment de hit-rate beïnvloedt.")
else:
    print("Geen tips gevonden om te testen.")


--- ANALYSE PER MARKTSEGMENT (Interactie P/E, B/M, Dividend) ---
       Market_Segment  Aantal Tips Hit-rate (Winst >3%)  \
0        Core / Mixed          268                51.1%   
1  Growth (Expansion)           92                50.0%   
2    Value (Maturity)           13                53.8%   

  Zachte Miss (-3% tot 3%) Harde Miss (Verlies <-3%)  
0                    21.3%                     27.6%  
1                    22.8%                     27.2%  
2                    15.4%                     30.8%  

--- CHI-KWADRAAT TOETS: SIGNIFICANTIE TUSSEN SEGMENTEN ---
Actual_Class        0.0  1.0  2.0
Market_Segment                   
Core / Mixed        137   57   74
Growth (Expansion)   46   21   25
Value (Maturity)      7    2    4

Chi-square statistic: 0.4008
p-waarde: 0.9824
RESULTAAT: Niet significant. Er is geen statistisch bewijs dat het segment de hit-rate beïnvloedt.


/tmp/ipykernel_4419/1659757990.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  segment_stats = analysis_df.groupby('Market_Segment').apply(get_stats).reset_index()


In [ ]:
# ---------------------------------------------------------
# 1.Profit for the model
# ---------------------------------------------------------
abs_diff_col = 'Return'

price_col = 'Close_Before'  #

backtest_df = test.copy()
backtest_df['Predicted_Class'] = y_test_pred_rf
backtest_df['Absolute_Difference'] = test[abs_diff_col].values
backtest_df['Starting_Price'] = test[price_col].values

# ---------------------------------------------------------
# 2. Rendement
# ---------------------------------------------------------
backtest_df['Real_ROI_Pct'] = (backtest_df['Absolute_Difference'] / backtest_df['Starting_Price']) * 100

backtest_df['Real_ROI_Pct'] = backtest_df['Real_ROI_Pct'].clip(lower=-100)



model_tips = backtest_df[backtest_df['Predicted_Class'] == 0]
gemiddeld_roi_model = model_tips['Real_ROI_Pct'].mean()

markt_roi_benchmark = backtest_df['Real_ROI_Pct'].mean()

# ---------------------------------------------------------
# 3. Resultats
# ---------------------------------------------------------
print("="*60)
print("=== FINALE BACKTEST RESULTATEN (PROCENTUEEL RENDEMENT) ===")
print("="*60)
print(f"Aantal koopsignalen van het model   : {len(model_tips)}")
print(f"Gemiddeld rendement per tip (Model) : {gemiddeld_roi_model:.2f}%")
print(f"Gemiddeld rendement markt (Benchmark): {markt_roi_benchmark:.2f}%")

outperformance = gemiddeld_roi_model - markt_roi_benchmark
print("-" * 60)
print(f"ALPHA (Outperformance)              : {outperformance:.2f}%")
print("="*60)

if outperformance > 0:
    print(f"SUCCES: Je model verslaat de markt met {outperformance:.2f}% per transactie!")
else:
    print("Helaas: Het model presteert minder dan de benchmark.")

# ---------------------------------------------------------
# 5. Reality Check
# ---------------------------------------------------------
print("\nTop 3 hoogste rendementen in tips (%):")
print(model_tips['Real_ROI_Pct'].nlargest(3))

=== FINALE BACKTEST RESULTATEN (PROCENTUEEL RENDEMENT) ===
Aantal koopsignalen van het model   : 373
Gemiddeld rendement per tip (Model) : 1.77%
Gemiddeld rendement markt (Benchmark): 0.27%
------------------------------------------------------------
ALPHA (Outperformance)              : 1.50%
SUCCES: Je model verslaat de markt met 1.50% per transactie!

Top 3 hoogste rendementen in tips (%):
8452     20.277158
12053    19.773457
4865     18.577362
Name: Real_ROI_Pct, dtype: float64


In [ ]:
# 1. What if you only buy the biggest chances
y_probs_final = rf_final_model_tuned.predict_proba(X_test_final)

analysis_df = test.copy()

analysis_df['Predicted_Class'] = y_test_pred_rf
analysis_df['Actual_Class'] = y_test.values  # Zorg dat y_test de werkelijke klassen bevat
analysis_df['Confidence_Kopen'] = y_probs_final[:, 0]

def conviction_level(prob):
    if prob >= 0.80: return 'High Conviction (>80%)'
    elif prob >= 0.60: return 'Medium Conviction (60-80%)'
    else: return 'Low Conviction (<60%)'

analysis_df['Conviction_Level'] = analysis_df['Confidence_Kopen'].apply(conviction_level)

tips_only = analysis_df[analysis_df['Predicted_Class'] == 0].copy()

if not tips_only.empty:
    print("\n--- CONVICTION ANALYSE: TUNED RF (29 FEATURES) ---")
    conviction_results = tips_only.groupby('Conviction_Level').apply(get_stats).reset_index()
    print(conviction_results)
else:
    print("Geen koopsignalen (Class 0) gevonden in de voorspellingen.")


--- CONVICTION ANALYSE: TUNED RF (29 FEATURES) ---
             Conviction_Level  Aantal Tips Hit-rate (Winst >3%)  \
0       Low Conviction (<60%)          337                49.9%   
1  Medium Conviction (60-80%)           36                61.1%   

  Zachte Miss (-3% tot 3%) Harde Miss (Verlies <-3%)  
0                    21.7%                     28.5%  
1                    19.4%                     19.4%  


/tmp/ipykernel_4419/937783664.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  conviction_results = tips_only.groupby('Conviction_Level').apply(get_stats).reset_index()


In [ ]:
# ---------------------------------------------------------
# 1. calculating rendement
# ---------------------------------------------------------

analysis_df['Actual_Return'] = test['Return'].values / test['Close_Before'].values


analysis_df['Actual_Return'] = analysis_df['Actual_Return'].clip(lower=-1.0, upper=1.0)

tips_only = analysis_df[analysis_df['Predicted_Class'] == 0].copy()


rendement_stats = tips_only.groupby('Conviction_Level')['Actual_Return'].agg(['mean', 'median', 'std', 'count']).reset_index()

rendement_stats.columns = ['Overtuiging', 'Gem. Rendement', 'Mediaan', 'Standaardafwijking', 'Aantal']


display_stats = rendement_stats.copy()
for col in ['Gem. Rendement', 'Mediaan', 'Standaardafwijking']:
    display_stats[col] = display_stats[col].map(lambda x: f"{x:.2%}")

print("\n--- CONVICTION ANALYSE (FINALE VERSIE) ---")
print(display_stats)

totaal_gemiddelde = tips_only['Actual_Return'].mean()
print(f"\nHet gemiddelde rendement over ALLE tips is nu: {totaal_gemiddelde:.2%}")


--- CONVICTION ANALYSE (FINALE VERSIE) ---
                  Overtuiging Gem. Rendement Mediaan Standaardafwijking  \
0       Low Conviction (<60%)          1.64%   2.40%              6.67%   
1  Medium Conviction (60-80%)          3.00%   2.80%              6.15%   

   Aantal  
0     337  
1      36  

Het gemiddelde rendement over ALLE tips is nu: 1.77%
